In [3]:
import openai
import os

from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

from langsmith import Client


### Download all data from qdrant

In [4]:
qdrant_client = QdrantClient(url="http://localhost:6333")


In [5]:
# Download all data from Qdrant

all_points = qdrant_client.scroll(
    collection_name="products",
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False
)


In [7]:
all_points[0][0].payload

{'description': 'Sensationnel Dashly lace front synthetic wigs have a wide hand tied swiss lace parting area with HD transparent lace and baby hair. Dashly wigs are prestyled yet customizable and easy to use.',
 'image': 'https://m.media-amazon.com/images/I/71APnrecqGL._SL1500_.jpg',
 'rating_number': 782,
 'price': 28.98,
 'average_rating': 4.0,
 'parent_asin': 'B07ZHP2WJX'}

In [8]:
all_context = [{"id": data.payload["parent_asin"], "text": data.payload["description"]} for data in all_points[0]]


In [9]:
all_context


[{'id': 'B07ZHP2WJX',
  'text': 'Sensationnel Dashly lace front synthetic wigs have a wide hand tied swiss lace parting area with HD transparent lace and baby hair. Dashly wigs are prestyled yet customizable and easy to use.'},
 {'id': 'B08XXQFF1Y',
  'text': 'Non Magnetic Eyeliner and Eyelashes Kit, Magic Self Adhesive False Eyelashes and Eyeliner, 5 Pairs 5D Reusable False Lashes with No Glue, Waterproof Lash Boxes with Mirror & Tweezers'},
 {'id': 'B00S1IADKK',
  'text': "If you're in need of a midday refresher, spray Invictus on yourself to liven up your day. This men's fragrance reveals a number of notes, including hints of grapefruit, Hedione jasmine, patchouli, bay leaves, and oak moss. Launched by Paco Rabanne in 2013, this delightful fragrance is perfect for the man who wants to always be at his best. This flexible scent can be worn at the office or at an afternoon gathering of friends or family."},
 {'id': 'B01K8QC0PI',
  'text': 'How to Use Apply nail polish on your nails an

### Render a prompt for generating synthetic eval reference dataset

In [10]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
             "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks.",
            },
            "question": {
                "type": "string",
                "description": "Suggested question.",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context.",
            },  
        },
    },
}


In [14]:
import json

SYSTEM_PROMPT = f"""
I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
I want you to come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multipple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>

I need to be able to parse the json output.
"""

USER_PROMPT = f"""
Here is the list of chunks, each list element is a dictionary with id and text:
{all_context}
"""


In [12]:
from groq import Groq
import os

groq_client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

In [20]:
response = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ]
)

print(response.choices[0].message.content)


Based on the provided chunks of text, I've generated 30 questions that could be answered using the context of the chunks. I've also included example answers, the IDs of the chunks that could be used to answer the questions, and the reasoning behind choosing those chunks.

Here are the 30 questions in JSON format:

```json
[
  {
    "reasoning": "The question asks about the material of the wig product, which is mentioned in chunk B0794MXH1S.",
    "question": "What is the material of the wig product?",
    "chunk_ids": ["B0794MXH1S"],
    "answer_example": "Kanekalon fiber"
  },
  {
    "reasoning": "The question asks about the features of the headband holder organizer, which is described in chunk B09CPCNQTT.",
    "question": "What are the features of the headband holder organizer?",
    "chunk_ids": ["B09CPCNQTT"],
    "answer_example": "It can hold 20-40 headbands, is made of acrylic, and has a simple installation method."
  },
  {
    "reasoning": "The question asks about the benefi

In [21]:
points = qdrant_client.scroll(
    collection_name="Amazon-items-collection-00",
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="parent_asin",
                match=MatchValue(value="B0BNVKS9WH")
            )
        ]
    ),
    limit=100,
    with_payload=True,
    with_vectors=False
)[0]


UnexpectedResponse: Unexpected Response: 404 (Not Found)
Raw response content:
b'{"status":{"error":"Not found: Collection `Amazon-items-collection-00` doesn\'t exist!"},"time":0.000441292}'

In [ ]:
points[0].payload
